In [0]:
%run ../../config/utils

In [0]:
import datetime
from pyspark.sql.window import Window
import pyspark.sql.functions as sqlf
import pandas as pd
from datetime import   datetime, timedelta
import numpy as np
from matplotlib import pyplot as plt # just for visualizations in debug mode
import mlflow

In [0]:
# Create a widget to configure the date used as "run date" , it defauls to today()
# This can be used to update or recreate past data. Beware the final saving step overwrites previously existing data for same run date. 

today_str = datetime.today().strftime("%Y-%m-%d")
dbutils.widgets.text("run_as_date", today_str, "Date for data processing")  # create the widget if missing
date_of_run_str = dbutils.widgets.get("run_as_date")
run_as_date     = datetime.strptime(date_of_run_str, "%Y-%m-%d").date() 

# calculate the date of the most recent saturday given the run date
recent_saturday = run_as_date - timedelta(days=(run_as_date.weekday() + 2) % 7)

print(f"Run as date: {run_as_date}")
print(f"Recent saturday: {recent_saturday}")

In [0]:
def get_features_list(features):
    continious_features     = features["continuous_features"].dropna().tolist()
    categorical_features    = features["categorical_features"].dropna().tolist()
    target_features         = features["target_features"].dropna().tolist()
    pk_features             = features["pk_features"].dropna().tolist()
    return categorical_features, continious_features, target_features, pk_features

#--------------------------------------------------------------------------------------------------------

# Build the column headers:

# Get a list of all the features
features = spark.read.csv( features_lookup_path, header=True) 
features = features.toPandas()

# These are used in cell [Transform data type based on feature type] to group columns by type ..
# or choose which columns to use on different scenarios
categorical_features, continious_features, target_feature, pk_features = get_features_list(features)

# Can be used to list the columns to be used in the model (see snippet below)
column_headers = categorical_features + continious_features + target_feature

In [0]:
try:
    # Use this if you want to test pulling data from your aws legacy process output instead of dbx (edit date in path as needed)
    # input_path = "s3://memberanalytics-data-out-prod/MODELDATA/BBM_PROPENSITY/ETL/INFERENCE_ETL/prod_2025-10-25/"  
    # input_data_raw = spark.read.parquet(input_path)

    input_data_raw = spark.table(model_bbm_etl_output).filter((sqlf.col('mode') == 'inference') & (sqlf.col('RUN_NAME') == recent_saturday.strftime('%Y-%m-%d')))

    
    # This filter reduces to 50% approx
    input_data_filtered = input_data_raw.filter(
        (sqlf.col('MBRSHP_STAT_CD') == 'AM') &
        (sqlf.col('LATEST_MBRSHP_EXP_DT') >= sqlf.current_date() - 95)
    )

    # take only required features 
    inference_data      = input_data_filtered.select( categorical_features + continious_features + pk_features )

    # inference_data_id   = input_data_filtered.select( pk_features ).toPandas()

    print("inference data loaded")
    

except Exception as e:
    print("Error loading inference data", e)
    dbutils.notebook.exit(f"Error loading inference data: {e}")

In [0]:
list_of_selected_features = ['LAST_EIGHT_WEEK_TRIPS',
                        'LAST_TWELVE_WEEK_TRIPS',
                        'LAST_TWENTY-SIX_WEEK_TRIPS',
                        'LAST_FIFTY-TWO_WEEK_TRIPS',
                        'LAST_FOUR_WEEK_DISTINCT_DAYS',
                        'LAST_EIGHT_WEEK_DISTINCT_DAYS',
                        'LAST_TWELVE_WEEK_DISTINCT_DAYS',
                        'LAST_TWELVE_WEEK_UNITS',
                        'LAST_FOUR_WEEK_TRANSACTIONS',
                        'LAST_TWELVE_WEEK_TRANSACTIONS',
                        'LAST_TWENTY-SIX_WEEK_TRANSACTIONS',
                        'DAYS_SINCE_LAST_TRIP',
                        'L8W_ATC_CPN_SPEND',
                        'L12W_ATC_CPN_SPEND',
                        'L26W_ATC_CPN_RED',
                        'L52W_ATC_CPN_RED',
                        'L52W_ATC_CPN_SPEND',
                        'DAYS_SINCE_LAST_COUPON_REDEEMED',
                        'LEIGHTW_COUPON_SAVINGS',
                        'LTWELVEW_COUPON_REDEMPTIONS',
                        'LTWELVEW_COUPON_SAVINGS',
                        'LTWENTY-SIXW_COUPON_REDEMPTIONS',
                        'LTWENTY-SIXW_COUPON_REDEMPTIONS_W_CLPLSS',
                        'LFIFTY-TWOW_COUPON_REDEMPTIONS',
                        'LFIFTY-TWOW_COUPON_REDEMPTIONS_W_CLPLSS',
                        'LFIFTY-TWOW_COUPON_SAVINGS',
                        'TENURE',
                        'DAYS_UNTIL_EXP',
                        'NUM_OF_RNWLS',
                        'L52W_PREFERRED_CLUB_TRIPS',
                        'member_age',
                        'SPEND_IN_STORE_BY_TRIPS_LAST_TWENTY-SIX_WEEKS',
                        'Non_Edible_Trips',
                        'Other_CPN_Trips',
                        'MEMBER_FREQUENCY_GROUP',
                        'HAS_QUOTIENT_ID',
                        'TENURE_GROUP',
                        'WEEK_OF_YEAR']

base_pd = inference_data.toPandas()

inference_data_id = base_pd[pk_features].copy()

X_selected = base_pd[list_of_selected_features].copy()

X_selected['TENURE_GROUP'] = X_selected['TENURE_GROUP'].apply(lambda x: 'tenured' if x == 'expired' else x)


# Transform data type based on feature type
# X_selected = (
#     pd.concat(
#         objs=[
#             X_selected[list(set(continious_features)  & set(X_selected.columns))].astype(float), 
#             X_selected[list(set(categorical_features) & set(X_selected.columns))].astype(str),
#         ],
#         axis=1,
#     )
# )

cont_cols = [c for c in continious_features if c in X_selected.columns]
cat_cols  = [c for c in categorical_features if c in X_selected.columns]

X_selected[cont_cols] = X_selected[cont_cols].astype(float)
X_selected[cat_cols]  = X_selected[cat_cols].astype(str)

X_selected = X_selected[cont_cols + cat_cols]

In [0]:
mlflow.set_registry_uri('databricks-uc')

data_preprocessing_model_uri    = f"models:/{data_preprocessing_model_catalog}@champion"
bbm_model_uri                   = f"models:/{bbm_model_catalog}@champion"

data_preprocessing  = mlflow.sklearn.load_model(data_preprocessing_model_uri)
bbm_model           = mlflow.xgboost.load_model(bbm_model_uri)

In [0]:
# Data preprocessing
X_mat = data_preprocessing.transform(X_selected)

# Run inference
score = bbm_model.predict_proba(X_mat)

# Enhance data with score
inference_data_id['score']= score[:, 1]

In [0]:
inference_data_id_sp = spark.createDataFrame(inference_data_id)

#-----------------------------------------------------------------------------------------------------------
# Combine with member info

mbr_ext = spark.table(silver_master_member_extended).select('MBRSHP_NBR','MBRSHP_SID')

inference_data_id_sp = inference_data_id_sp.join(mbr_ext, "MBRSHP_SID", "inner").withColumn("new_decile",
        sqlf.ntile(10).over(Window.partitionBy().orderBy(sqlf.col("score").desc())))

#------------------------------------------------------------------------------------------------------------

inference_data_id_pd = inference_data_id_sp.toPandas()

inference_data_id_pd.columns        = ['MBRSHP_SID','end_date','score','mbrshp_nbr','decile']
inference_data_id_pd['end_date']    = pd.to_datetime(inference_data_id_pd['end_date'])
inference_data_id_pd['mbrshp_nbr']  = inference_data_id_pd['mbrshp_nbr'].str.zfill(11)#.astype(int)
inference_data_id_pd['end_date']    = inference_data_id_pd['end_date'].dt.strftime('%d-%b-%y')
inference_data_id_pd['score']       = inference_data_id_pd['score'].round(4)

output = inference_data_id_pd[['decile','mbrshp_nbr','score','end_date']]

In [0]:
# Turn panda into spark df
output_sp = spark.createDataFrame(output)

# Add run_date and run_name(last_saturday) columns 
df_w_additional_columns = output_sp.select(
    "*",
    sqlf.lit(run_as_date).cast("date").alias("run_date"),
    sqlf.lit(recent_saturday).cast("string").alias("run_name")
)

# Save in overwrite mode for specific columns
df_w_additional_columns.write.mode("overwrite").option(
    "replaceWhere",
   f"run_name = '{recent_saturday}' "
).saveAsTable(model_bbm_inference_output)

In [0]:
# We will save to DBX Volume by default. If in prod we additionally move such file to s3
filename = f"MKT_EDW_BBM_SCORE_{recent_saturday.strftime('%Y%m%d')}.csv"

file_path_in_volume = f'{bbm_score_volume_path}/{filename}'

output.to_csv(file_path_in_volume, index=False, header=True)
print(f"Output successfully saved to volume: {file_path_in_volume}.")

# LL Note: we recommend EDW team to access this directly from UnityCatalog and delete this step and their related files/ resources

if environment == 'prod': # store scores in s3 at the same locations the legacy process did
    try:

        dbutils.fs.cp(file_path_in_volume, f"{bbm_score_path}/{filename}") # just copy the 1st one so we can use 'move' in next step
        dbutils.fs.mv(file_path_in_volume, f"{bbm_score_edw_path}/{filename}")
        print(f"Inference output saved to additional S3 locations ({bbm_score_path}/{filename} and {bbm_score_edw_path}/{filename}) successfully.")

    except Exception as e:
        print(f"Error saving results to S3: {e}")  

In [0]:
# Script to compare DBX score distribution output vs legacy output while debugging

# output_date = '20251025'#'20250809'  recent_saturday.strftime('%Y%m%d')
# old_file_path =  f"s3://memberanalytics-data-out-prod/MODELDATA/BBM_PROPENSITY/BBM_SALES_SCORE/MKT_EDW_BBM_SCORE_{output_date}.csv"
# output_old = spark.read.csv(old_file_path, header=True, inferSchema=True).toPandas()

# df1 = output
# df2 = output_old
# df1['score'].plot.hist(bins=50, alpha=0.5, label='New')
# df2['score'].plot.hist(bins=50, alpha=0.5, label='Legacy')
# plt.xlabel('Score')
# plt.ylabel('Frequency')
# plt.title('Score Distribution Comparison')
# plt.legend()
# plt.show()